# Paper figures — block-DDA_Py × block-VIEM.jl

Generates the canonical paper figures from the v0.7.6 sweep results.
Reads byte-compatible HDF5 files from both solver repos:

- DDA side: `~/Python/block-DDA_Py/dda_results/paper/{shape}_{material}.hdf5`
- VIEM side: `~/Julia/block-VIEM.jl/viem_results/paper/{shape}_{material}.hdf5`
- MSTM (doublet exact): `~/Julia/block-VIEM.jl/viem_results/paper/mstm_doublet_{material}.hdf5`

**Sections**
1. Optical cross-sections — Q_ext, Q_abs, Q_sca vs a_eq
2. CAS-v2 forward/backward amplitudes — |⟨S_fw_θ⟩|, |⟨S_fw_φ⟩|, |⟨S_bk⟩|
3. DDA ↔ VIEM relative error
4. dpl / lc convergence study
5. Block-Krylov RHS scaling
6. Cost summary table

Robust to missing VIEM/MSTM data: each figure overlays whichever sources are available.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---- paths ----------------------------------------------------------------
DDA_DIR  = Path(os.environ.get('DDA_DIR',  '~/Python/block-DDA_Py/dda_results/paper')).expanduser()
VIEM_DIR = Path(os.environ.get('VIEM_DIR', '~/Julia/block-VIEM.jl/viem_results/paper')).expanduser()

assert DDA_DIR.is_dir(), f'DDA_DIR not found: {DDA_DIR}'
if not VIEM_DIR.is_dir():
    warnings.warn(f'VIEM_DIR not found: {VIEM_DIR} — VIEM/MSTM curves will be skipped.')

# ---- paper grid -----------------------------------------------------------
SHAPES    = ['sphere', 'oblate', 'gre', 'doublet']
MATERIALS = ['n15', 'n20', 'Au']

MATERIAL_LABEL = {
    'n15': r'$n_p=1.5+0.01j$',
    'n20': r'$n_p=2.0+0.0j$',
    'Au':  r'Au (J&C 1972)',
}
SHAPE_LABEL = {
    'sphere':  'sphere',
    'oblate':  'oblate (a:b:c = 3:3:1)',
    'gre':     r'GRE ($\beta_{gre}=0.2$)',
    'doublet': 'doublet (gap=0.1R)',
}

# ---- style ----------------------------------------------------------------
STYLE = {
    'dda':  dict(marker='o', mfc='none', linestyle='-',  color='C0', label='DDA'),
    'viem': dict(marker='s', mfc='none', linestyle='--', color='C1', label='VIEM'),
    'mie':  dict(marker=None,            linestyle=':',  color='k',  label='Mie (sphere)'),
    'mstm': dict(marker='x',             linestyle='-.', color='C3', label='MSTM (doublet)'),
}

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
})

OUT_DIR = DDA_DIR / 'figures'
OUT_DIR.mkdir(exist_ok=True)
print(f'DDA_DIR  = {DDA_DIR}')
print(f'VIEM_DIR = {VIEM_DIR}  (exists={VIEM_DIR.is_dir()})')
print(f'OUT_DIR  = {OUT_DIR}')

In [ ]:
# ---- HDF5 loaders ---------------------------------------------------------
def _squeeze_rv_orient(arr):
    """Sweep arrays have shape (1,1,N_rv,1,1,1,100) for observables, or
    (1,1,N_rv,1,1,1) for Mie. Return either (N_rv, 100) or (N_rv,)."""
    sq = np.squeeze(arr)
    if sq.ndim == 0:  # single (N_rv=1) Mie scalar — promote to (1,)
        sq = sq[np.newaxis]
    return sq

def load_paper(path):
    """Read /target/simulated_data + /target/cost from a paper HDF5 file.

    Returns dict with keys:
      r_v_base, r_ve     — (N_rv,)
      C_ext, C_abs       — (N_rv, 100)         [μm²]
      S_fw_theta, S_fw_phi, S_bk — (N_rv, 100) [μm], complex
      C_ext_mie, C_abs_mie       — (N_rv,)
      S_fw_mie, S_bk_mie         — (N_rv,) complex
      cost: dict of cost-group arrays squeezed to (N_rv,)
    """
    path = Path(path)
    with h5py.File(path, 'r') as f:
        t = f['target']
        sim = t['simulated_data']
        out = {
            'r_v_base':   t['r_v_base_list'][:].ravel(),
            'r_ve':       _squeeze_rv_orient(sim['r_ve'][:]),
            'C_ext':      _squeeze_rv_orient(sim['C_ext'][:]),
            'C_abs':      _squeeze_rv_orient(sim['C_abs'][:]),
            'S_fw_theta': _squeeze_rv_orient(sim['S_fw_PCAS_theta'][:]),
            'S_fw_phi':   _squeeze_rv_orient(sim['S_fw_PCAS_phi'][:]),
            'S_bk':       _squeeze_rv_orient(sim['S_bk_OCBS'][:]),
            'C_ext_mie':  _squeeze_rv_orient(sim['C_ext_mie'][:]),
            'C_abs_mie':  _squeeze_rv_orient(sim['C_abs_mie'][:]),
            'S_fw_mie':   _squeeze_rv_orient(sim['S_fw_PCAS_mie'][:]),
            'S_bk_mie':   _squeeze_rv_orient(sim['S_bk_OCBS_mie'][:]),
        }
        if 'cost' in t:
            c = t['cost']
            out['cost'] = {k: np.squeeze(c[k][:]) for k in c.keys()
                           if c[k].ndim >= 1 and c[k].shape != ()}
    return out

def try_load_dda(shape, material):
    p = DDA_DIR / f'{shape}_{material}.hdf5'
    return load_paper(p) if p.is_file() else None

def try_load_viem(shape, material):
    p = VIEM_DIR / f'{shape}_{material}.hdf5'
    return load_paper(p) if p.is_file() else None

def try_load_mstm(material):
    p = VIEM_DIR / f'mstm_doublet_{material}.hdf5'
    if not p.is_file():
        return None
    with h5py.File(p, 'r') as f:
        t = f['target']
        sim = t['simulated_data']
        return {
            'r_v_base':   t['r_v_base_list'][:].ravel(),
            'C_ext':      _squeeze_rv_orient(sim['C_ext'][:]),
            'C_abs':      _squeeze_rv_orient(sim['C_abs'][:]),
            'S_fw_theta': _squeeze_rv_orient(sim['S_fw_PCAS_theta'][:]),
            'S_fw_phi':   _squeeze_rv_orient(sim['S_fw_PCAS_phi'][:]),
            'S_bk':       _squeeze_rv_orient(sim['S_bk_OCBS'][:]),
        }

# Convenience: orientation averages
def Q_orient_avg(C, a_eq):
    """<Q>_orient with C of shape (N_rv, 100), a_eq of shape (N_rv,).
    Returns (N_rv,)."""
    Q = C / (np.pi * a_eq[:, None]**2)
    return np.nanmean(Q, axis=-1)

def S_orient_mag(S):
    """<|S|>_orient with S of shape (N_rv, 100). Returns (N_rv,) real."""
    return np.nanmean(np.abs(S), axis=-1)

def S_avg_complex(S):
    """<S>_orient (complex). Returns (N_rv,) complex."""
    return np.nanmean(S, axis=-1)

# Test load on one DDA file
test = try_load_dda('sphere', 'n20')
if test is not None:
    print('sphere_n20: r_v_base =', test['r_v_base'])
    print('   C_ext shape =', test['C_ext'].shape)
    print('   <Q_ext>_orient =', Q_orient_avg(test['C_ext'], test['r_ve']))

## 1. Optical cross-sections — $Q_{\rm ext}$, $Q_{\rm abs}$, $Q_{\rm sca}$

$Q_X = \langle C_X \rangle_{\rm orient} / (\pi a_{\rm eq}^2)$, with $a_{\rm eq}$ the volume-equivalent radius.

Layout: 4 shapes (rows) × 3 materials (cols) = 12 panels per quantity.
Reference overlay: Mie for `sphere`, MSTM for `doublet` (when available).

In [ ]:
def plot_Q_grid(quantity='ext', save=True):
    """quantity ∈ {'ext', 'abs', 'sca'}."""
    fig, axes = plt.subplots(len(SHAPES), len(MATERIALS),
                              figsize=(11, 12), sharex=True, sharey='row')
    fig.suptitle(rf'$Q_{{{quantity}}}$  vs  $a_{{\rm eq}}$', y=0.995)
    for i, shape in enumerate(SHAPES):
        for j, mat in enumerate(MATERIALS):
            ax = axes[i, j]
            dda  = try_load_dda(shape, mat)
            viem = try_load_viem(shape, mat)
            mstm = try_load_mstm(mat) if shape == 'doublet' else None
            for src, dat in [('dda', dda), ('viem', viem)]:
                if dat is None:
                    continue
                a = dat['r_ve']
                C_ext = Q_orient_avg(dat['C_ext'], a)
                C_abs = Q_orient_avg(dat['C_abs'], a)
                if quantity == 'ext':
                    Y = C_ext
                elif quantity == 'abs':
                    Y = C_abs
                elif quantity == 'sca':
                    Y = C_ext - C_abs
                ax.plot(a, Y, **STYLE[src])
            # Mie reference (sphere only)
            if shape == 'sphere' and dda is not None:
                a = dda['r_v_base']
                C_ext_m = dda['C_ext_mie'] / (np.pi * a**2)
                C_abs_m = dda['C_abs_mie'] / (np.pi * a**2)
                Q_m = {'ext': C_ext_m, 'abs': C_abs_m, 'sca': C_ext_m - C_abs_m}[quantity]
                ax.plot(a, Q_m, **STYLE['mie'])
            # MSTM reference (doublet only)
            if mstm is not None:
                a = mstm['r_v_base']
                C_ext_x = np.nanmean(np.abs(mstm['C_ext']), axis=-1) if mstm['C_ext'].ndim==2 else mstm['C_ext']
                C_abs_x = np.nanmean(np.abs(mstm['C_abs']), axis=-1) if mstm['C_abs'].ndim==2 else mstm['C_abs']
                Q_x = {'ext': C_ext_x/(np.pi*a**2), 'abs': C_abs_x/(np.pi*a**2),
                       'sca': (C_ext_x-C_abs_x)/(np.pi*a**2)}[quantity]
                ax.plot(a, Q_x, **STYLE['mstm'])
            ax.set_xscale('log')
            ax.set_yscale('log')
            if i == 0:
                ax.set_title(MATERIAL_LABEL[mat])
            if j == 0:
                ax.set_ylabel(SHAPE_LABEL[shape] + f'\n$Q_{{{quantity}}}$')
            if i == len(SHAPES) - 1:
                ax.set_xlabel(r'$a_{\rm eq}$ [μm]')
    # global legend
    handles = [Line2D([], [], **{k:v for k,v in s.items() if k!='label'}, label=s['label'])
               for s in STYLE.values()]
    fig.legend(handles=handles, ncol=4, loc='lower center', bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout()
    if save:
        out = OUT_DIR / f'fig1_Q_{quantity}.png'
        fig.savefig(out, bbox_inches='tight')
        print('saved', out)
    plt.show()

for q in ('ext', 'abs', 'sca'):
    plot_Q_grid(q)

## 2. CAS-v2 forward / backward amplitudes

Plots the magnitude of the orientation-averaged complex amplitude
$|\langle S \rangle_{\rm orient}|$ for

- $S_{\rm fw}^{(\theta)} = S_{11}(0) + iS_{12}(0)$ — forward, $\theta$-component
- $S_{\rm fw}^{(\phi)} = S_{22}(0) - iS_{21}(0)$ — forward, $\phi$-component
- $S_{\rm bk} = (S_{11}+S_{22}+iS_{12}-iS_{21})(180°) / \sqrt{2}$ — backward

These reduce to the diagonal CAS-v2 inputs that the optical-counter forward
model consumes (Mishchenko 2000 + BH83 sign convention; see
`docs/paper_simulation_conditions_dda.md` §8.4).

Two complementary views per panel:
- **|⟨S⟩|**: coherent orientation-average magnitude
- **⟨|S|⟩**: incoherent magnitude average (for comparison)

In [ ]:
S_KEYS = [
    ('S_fw_theta', r'|\langle S_{\rm fw}^{(\theta)}\rangle|', 'S_fw_mie'),
    ('S_fw_phi',   r'|\langle S_{\rm fw}^{(\phi)}\rangle|',   'S_fw_mie'),
    ('S_bk',       r'|\langle S_{\rm bk}\rangle|',             'S_bk_mie'),
]

def plot_S_grid(s_key, latex, mie_key, mode='coherent', save=True):
    """mode ∈ {'coherent', 'incoherent'} — |<S>| vs <|S|>."""
    title_suffix = '|⟨S⟩|' if mode == 'coherent' else '⟨|S|⟩'
    fig, axes = plt.subplots(len(SHAPES), len(MATERIALS),
                              figsize=(11, 12), sharex=True, sharey='row')
    fig.suptitle(f'$\\mathbf{{{title_suffix}}}$ for ${latex}$  vs  $a_{{\\rm eq}}$', y=0.995)
    for i, shape in enumerate(SHAPES):
        for j, mat in enumerate(MATERIALS):
            ax = axes[i, j]
            dda  = try_load_dda(shape, mat)
            viem = try_load_viem(shape, mat)
            mstm = try_load_mstm(mat) if shape == 'doublet' else None
            for src, dat in [('dda', dda), ('viem', viem)]:
                if dat is None:
                    continue
                S = dat[s_key]
                Y = np.abs(S_avg_complex(S)) if mode == 'coherent' else S_orient_mag(S)
                ax.plot(dat['r_ve'], Y, **STYLE[src])
            if shape == 'sphere' and dda is not None:
                a = dda['r_v_base']
                ax.plot(a, np.abs(dda[mie_key]), **STYLE['mie'])
            if mstm is not None:
                a = mstm['r_v_base']
                S_x = mstm[s_key]
                Y_x = (np.abs(S_avg_complex(S_x)) if mode == 'coherent' and S_x.ndim==2
                       else (S_orient_mag(S_x) if S_x.ndim==2 else np.abs(S_x)))
                ax.plot(a, Y_x, **STYLE['mstm'])
            ax.set_xscale('log')
            ax.set_yscale('log')
            if i == 0:
                ax.set_title(MATERIAL_LABEL[mat])
            if j == 0:
                ax.set_ylabel(f'{SHAPE_LABEL[shape]}\n${latex}$ [μm]')
            if i == len(SHAPES) - 1:
                ax.set_xlabel(r'$a_{\rm eq}$ [μm]')
    handles = [Line2D([], [], **{k:v for k,v in s.items() if k!='label'}, label=s['label'])
               for s in STYLE.values()]
    fig.legend(handles=handles, ncol=4, loc='lower center', bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout()
    if save:
        out = OUT_DIR / f'fig2_{s_key}_{mode}.png'
        fig.savefig(out, bbox_inches='tight')
        print('saved', out)
    plt.show()

for s_key, latex, mie_key in S_KEYS:
    plot_S_grid(s_key, latex, mie_key, mode='coherent')
    plot_S_grid(s_key, latex, mie_key, mode='incoherent')

## 3. DDA ↔ VIEM relative error

$\varepsilon_X(a_{\rm eq}) = |X_{\rm DDA} - X_{\rm VIEM}| / |X_{\rm VIEM}|$,
with both sides orientation-averaged the same way (coherent for amplitudes,
real for cross-sections).

Au columns will show 1.0 since DDA stagnates and the reference is taken from VIEM.

In [ ]:
OBSERVABLES = [
    ('Q_ext',      lambda d: Q_orient_avg(d['C_ext'], d['r_ve'])),
    ('Q_abs',      lambda d: Q_orient_avg(d['C_abs'], d['r_ve'])),
    ('|<S_fw_θ>|', lambda d: np.abs(S_avg_complex(d['S_fw_theta']))),
    ('|<S_fw_φ>|', lambda d: np.abs(S_avg_complex(d['S_fw_phi']))),
    ('|<S_bk>|',   lambda d: np.abs(S_avg_complex(d['S_bk']))),
]

def fig_relative_error(save=True):
    fig, axes = plt.subplots(len(SHAPES), len(MATERIALS),
                              figsize=(11, 12), sharex=True)
    fig.suptitle(r'$\varepsilon = |X_{\rm DDA} - X_{\rm VIEM}|/|X_{\rm VIEM}|$ vs $a_{\rm eq}$', y=0.995)
    for i, shape in enumerate(SHAPES):
        for j, mat in enumerate(MATERIALS):
            ax = axes[i, j]
            dda  = try_load_dda(shape, mat)
            viem = try_load_viem(shape, mat)
            if dda is None or viem is None:
                ax.text(0.5, 0.5, 'no data', transform=ax.transAxes, ha='center', va='center')
            else:
                a = dda['r_ve']
                for label, fn in OBSERVABLES:
                    yd, yv = fn(dda), fn(viem)
                    err = np.abs(yd - yv) / np.where(np.abs(yv) > 0, np.abs(yv), np.nan)
                    ax.plot(a, err, marker='o', markersize=3, label=label)
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.axhline(1e-2, color='gray', lw=0.5, ls=':')
            if i == 0: ax.set_title(MATERIAL_LABEL[mat])
            if j == 0: ax.set_ylabel(f'{SHAPE_LABEL[shape]}\nrel. err.')
            if i == len(SHAPES)-1: ax.set_xlabel(r'$a_{\rm eq}$ [μm]')
    axes[0, -1].legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
    plt.tight_layout()
    if save:
        out = OUT_DIR / 'fig3_dda_viem_rel_error.png'
        fig.savefig(out, bbox_inches='tight')
        print('saved', out)
    plt.show()

fig_relative_error()

## 4. dpl / lc convergence study

Runs at $a_{\rm eq}=0.1\,\mu m$, single ZYZ-identity orientation.

- DDA: dpl ∈ {10, 14, 17, 24, 34} (paper-fixed grid)
- VIEM: lc factor ∈ {1.5, 1.0, 0.7, 0.5, 0.35}

Both grids are aligned coarse → fine. Plot $|X(\rm fine) - X(\rm coarsest)|$
or $X$ itself; here we show normalized $X / X(\rm finest)$.

In [ ]:
DPL_LIST = [10, 14, 17, 24, 34]
LC_LIST  = [1.5, 1.0, 0.7, 0.5, 0.35]

def load_dpl_convergence(shape, material):
    p = DDA_DIR / f'convergence_{shape}_{material}.hdf5'
    if not p.is_file():
        return None
    with h5py.File(p, 'r') as f:
        g = f['target/dpl_convergence']
        return {k: g[k][:] for k in g.keys()}

def load_lc_convergence(shape, material):
    p = VIEM_DIR / f'convergence_{shape}_{material}.hdf5'
    if not p.is_file():
        return None
    with h5py.File(p, 'r') as f:
        # adjust group name if VIEM uses a different one
        for grp in ('target/lc_convergence', 'target/lc_convergence_factor'):
            if grp in f:
                g = f[grp]
                return {k: g[k][:] for k in g.keys()}
    return None

CONV_SHAPES = ['sphere', 'oblate', 'gre']  # doublet not in §11.4 scope
OBS_NAMES   = ['C_ext', 'C_abs', 'S_fw_theta', 'S_fw_phi', 'S_bk']

def fig_dpl_convergence(save=True):
    fig, axes = plt.subplots(len(CONV_SHAPES), len(MATERIALS),
                              figsize=(11, 9), sharex=True)
    fig.suptitle(r'dpl-convergence (DDA) / lc-convergence (VIEM) at $a_{\rm eq}=0.1\,\mu m$', y=0.995)
    for i, shape in enumerate(CONV_SHAPES):
        for j, mat in enumerate(MATERIALS):
            ax = axes[i, j]
            dda = load_dpl_convergence(shape, mat)
            viem = load_lc_convergence(shape, mat)
            if dda is not None:
                # dpl_list dataset is the actual dpl values; observables vary by shape
                dpl = dda.get('dpl_values', dda.get('dpl', np.array(DPL_LIST)))
                Y = None
                for key in ('C_ext', 'Q_ext', 'cext'):
                    if key in dda:
                        Y = dda[key]; break
                if Y is not None and np.isfinite(Y).any():
                    Y = Y / np.nanmean(Y[np.isfinite(Y)][-1:])  # normalize to finest
                    ax.plot(np.asarray(dpl).ravel(), Y.ravel(),
                            'o-', color='C0', label='DDA C_ext')
            if viem is not None:
                lc = viem.get('lc_factors', viem.get('lc_values', np.array(LC_LIST)))
                Y = None
                for key in ('C_ext', 'Q_ext', 'cext'):
                    if key in viem:
                        Y = viem[key]; break
                if Y is not None and np.isfinite(Y).any():
                    Y = Y / np.nanmean(Y[np.isfinite(Y)][-1:])
                    # plot vs lc on twin x for direct visual comparison
                    ax2 = ax.twiny()
                    ax2.plot(np.asarray(lc).ravel(), Y.ravel(),
                             's--', color='C1', label='VIEM C_ext')
                    ax2.set_xlabel('lc factor (VIEM)')
                    ax2.invert_xaxis()  # finer -> right, like dpl
            ax.set_xscale('log')
            if i == 0: ax.set_title(MATERIAL_LABEL[mat])
            if j == 0: ax.set_ylabel(f'{SHAPE_LABEL[shape]}\n$X / X_{{\\rm finest}}$')
            if i == len(CONV_SHAPES)-1: ax.set_xlabel('dpl (DDA)')
    plt.tight_layout()
    if save:
        out = OUT_DIR / 'fig4_dpl_lc_convergence.png'
        fig.savefig(out, bbox_inches='tight')
        print('saved', out)
    plt.show()

fig_dpl_convergence()

## 5. Block-Krylov RHS scaling

$L \in \{1, 2, 4, 8, 16, 32, 64, 128\}$ for sphere × {n15, n20, Au}.

Plotted: iter count and end-to-end wall per orientation as functions of $L$.

In [ ]:
def load_rhs_scaling(side, material):
    base = DDA_DIR if side == 'dda' else VIEM_DIR
    p = base / f'sphere_{material}.hdf5'
    if not p.is_file():
        return None
    with h5py.File(p, 'r') as f:
        if 'target/rhs_scaling' not in f:
            return None
        g = f['target/rhs_scaling']
        out = {'L_values': g['L_values'][:],
               'n_occ':    g['n_occ'][:] if 'n_occ' in g else g['n_dof'][:]}
        # method subgroup — gmres for both sides (v0.7.6)
        for m in ('gmres', 'GMRES'):
            if m in g:
                sub = g[m]
                out[m] = {k: sub[k][:] for k in ('iters', 'converged', 't_total_s', 't_end2end_per_orient_s') if k in sub}
        return out

def fig_rhs_scaling(save=True):
    fig, axes = plt.subplots(2, len(MATERIALS), figsize=(11, 7), sharex=True)
    fig.suptitle('Block-GMRES scaling on sphere × {n15, n20, Au}', y=0.99)
    for j, mat in enumerate(MATERIALS):
        for side, color in [('dda', 'C0'), ('viem', 'C1')]:
            d = load_rhs_scaling(side, mat)
            if d is None: continue
            L = d['L_values']
            for s_idx in range(d['n_occ'].shape[0]):  # iterate over a_eq slots
                if 'gmres' not in d: continue
                iters = d['gmres']['iters'][:, s_idx, 0, 0, 0] if d['gmres']['iters'].ndim == 5 else d['gmres']['iters'][:, s_idx]
                t_per = d['gmres']['t_end2end_per_orient_s'][:, s_idx, 0, 0, 0] if d['gmres']['t_end2end_per_orient_s'].ndim == 5 else d['gmres']['t_end2end_per_orient_s'][:, s_idx]
                axes[0, j].plot(L, iters, marker='o', color=color, alpha=0.4 + 0.15*s_idx,
                                label=f'{side.upper()} a_eq idx {s_idx}' if j == 0 else None)
                axes[1, j].plot(L, t_per, marker='o', color=color, alpha=0.4 + 0.15*s_idx)
        axes[0, j].set_title(MATERIAL_LABEL[mat])
        axes[0, j].set_ylabel('GMRES iters' if j == 0 else '')
        axes[1, j].set_xlabel('L (RHS count)')
        axes[1, j].set_ylabel('t / orient [s]' if j == 0 else '')
        for ax in (axes[0, j], axes[1, j]):
            ax.set_xscale('log', base=2); ax.set_yscale('log')
    axes[0, 0].legend(fontsize=7, loc='best')
    plt.tight_layout()
    if save:
        out = OUT_DIR / 'fig5_rhs_scaling.png'
        fig.savefig(out, bbox_inches='tight')
        print('saved', out)
    plt.show()

fig_rhs_scaling()

## 6. Cost summary table

Per-file totals of t_total_s, peak_rss_bytes, converged-slot count.
Mirrors `dda_results/paper/cost_estimates.md`.

In [ ]:
def cost_summary_table():
    rows = []
    for shape in SHAPES:
        for mat in MATERIALS:
            d = try_load_dda(shape, mat)
            if d is None or 'cost' not in d:
                continue
            c = d['cost']
            t_total_min = float(np.sum(c['t_total_s'])) / 60
            peak_GB     = float(np.max(c['peak_rss_bytes'])) / 1024**3
            n_conv      = int(np.sum(c['converged']))
            n_slot      = int(c['converged'].size)
            iters_max   = int(np.max(c['iters']))
            rows.append((f'{shape}_{mat}', n_conv, n_slot, t_total_min, peak_GB, iters_max))
    print(f"{'file':<14} {'conv':<7} {'wall (min)':<12} {'peak RSS (GB)':<15} {'max iters':<10}")
    print('-' * 60)
    tot_t, tot_rss = 0.0, 0.0
    for name, nc, ns, t, rss, mi in rows:
        print(f'{name:<14} {nc}/{ns:<5} {t:<12.2f} {rss:<15.2f} {mi}')
        tot_t += t; tot_rss = max(tot_rss, rss)
    print('-' * 60)
    print(f"{'TOTAL':<14} {'':<7} {tot_t:<12.2f} (peak {tot_rss:.2f})")

cost_summary_table()

## Next steps

- VIEM HDF5 files need to be available under `VIEM_DIR` for full DDA↔VIEM comparison
- MSTM doublet reference (`mstm_doublet_{material}.hdf5`) needed for Fig 1/2 doublet rows
- Refine paper styling (font, palette, spacing) once content is final
- Optional: export each figure as `.pdf` for LaTeX inclusion (`fig.savefig('fig1.pdf')`)
- Optional: split orientation-resolved phase/intensity views into a supplementary notebook